# Сабмит public test

Полный public split → адаптивные 150 кандидатов → обученный CatBoost → 20 разных треков и короткий ответ про первый. Метки public test не используются. Текст ответа формируется прозрачными шаблонами из метаданных и истории, без генеративной модели.

## Модель и настройки

In [1]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["OMP_NUM_THREADS"] = "6"
import gc, re, json, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import bm25s, joblib
from scipy.sparse import load_npz, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from IPython.display import display

SEED = 42
VALID_LIMIT = 4000
TEST_LIMIT = None
BATCH_SIZE = 64
MAX_K = 1000
BUDGETS = [100, 200, 500, 1000]
# Работает при запуске из корня проекта или папки датасета.
PROJECT_DIR = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "music4all_crs" / "train_queries.parquet").exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Не найдена папка music4all_crs с исходными данными")
DATA_DIR = PROJECT_DIR / "music4all_crs"
BASE_DIR = PROJECT_DIR / "retrieval_artifacts"
SEARCH_DIR = BASE_DIR
CACHE_DIR = SEARCH_DIR / "candidate_cache"
OUT_DIR = PROJECT_DIR / "adaptive150_catboost_artifacts"
OUT_DIR.mkdir(exist_ok=True)

from catboost import CatBoostRanker, Pool
from sklearn.feature_extraction.text import CountVectorizer

TRAIN_LIMIT = 30000
ITERATIONS = 350
THREADS = 6
CANDIDATE_LIMIT = 150
POLICY_DIR = OUT_DIR
CAP_CONFIG = json.loads((POLICY_DIR / "candidate_policy.json").read_text())
TOP_K = 1000
RRF_CONSTANT = 60
SOURCE_NAMES = json.loads((SEARCH_DIR / "generators.json").read_text())[
    "generators"
]
OLD_FEATURES = json.loads((BASE_DIR / "base_feature_names.json").read_text())
FEATURES = (
    OLD_FEATURES
    + [f"{name}_source_rank" for name in SOURCE_NAMES]
    + [f"{name}_source_hit" for name in SOURCE_NAMES]
    + ["all_source_count", "all_rrf_score", "selection_rrf_score", "selection_rank"]
)
CAT_FEATURES = ["query_type"]
NUMERIC_FEATURES = [f for f in FEATURES if f not in CAT_FEATURES]
print("Features:", len(FEATURES), "generators:", len(SOURCE_NAMES))

import pyarrow as pa
import pyarrow.parquet as pq
from itertools import islice

SUBMIT_DIR = PROJECT_DIR / "submission_public_artifacts"
SUBMIT_DIR.mkdir(exist_ok=True)
PARTS_DIR = SUBMIT_DIR / "parts"
PARTS_DIR.mkdir(exist_ok=True)
SUBMISSION_PATH = PROJECT_DIR / "submission_public.parquet"
model = CatBoostRanker()
model.load_model(str(OUT_DIR / "catboost_ranker.cbm"))
assert model.feature_names_ == FEATURES
# Снимок политики, использованной именно при обучении модели.
CAP_CONFIG = json.loads((OUT_DIR / "metrics.json").read_text())["config"]["cap_config"]
(OUT_DIR / "candidate_policy.json").write_text(json.dumps(CAP_CONFIG, indent=2))


# Контрольные точки можно переиспользовать только с той же моделью и входами.
def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


run_signature = {
    "model_sha256": sha256_bytes((OUT_DIR / "catboost_ranker.cbm").read_bytes()),
    "queries_sha256": sha256_bytes(
        (DATA_DIR / "test_public_queries.parquet").read_bytes()
    ),
    "policy_sha256": sha256_bytes(json.dumps(CAP_CONFIG, sort_keys=True).encode()),
    "features_sha256": sha256_bytes(json.dumps(FEATURES).encode()),
    "response_version": 2,
}
manifest = SUBMIT_DIR / "run_config.json"
if manifest.exists():
    assert (
        json.loads(manifest.read_text()) == run_signature
    ), "Модель, запросы или политика изменились. Выберите новую SUBMIT_DIR для нового сабмита."
else:
    manifest.write_text(json.dumps(run_signature, indent=2))


Features: 58 generators: 11


## Все запросы public test

In [2]:
def normalize(text):
    return " ".join(re.findall(r"\w+", str(text).casefold()))


def text_bucket(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16) % 10


def sample_queries(frame, limit):
    if limit is None or len(frame) <= limit:
        return frame.reset_index(drop=True)
    # Случайная выборка сохраняет естественное распределение типов.
    return frame.sample(n=limit, random_state=SEED).reset_index(drop=True)


train_inputs = pd.read_parquet(DATA_DIR / "train_queries.parquet")
test_inputs = pd.read_parquet(DATA_DIR / "test_public_queries.parquet")
assert train_inputs.query_id.is_unique and test_inputs.query_id.is_unique
train_inputs["normalized"] = train_inputs["query"].map(normalize)
test_texts = set(test_inputs["query"].map(normalize))
buckets = train_inputs.normalized.map(text_bucket)
index_inputs = train_inputs[
    (buckets < 6) & ~train_inputs.normalized.isin(test_texts)
].copy()
print("Public queries:", len(test_inputs))
assert len(test_inputs) == 273137


Public queries: 273137


## Метаданные и обучающие документы

In [3]:
META_COLUMNS = [
    "m4a_id",
    "m4a_artist",
    "m4a_song",
    "m4a_album",
    "m4a_genres_full",
    "m4a_tags_full",
    "pseudo_caption",
    "lyrics",
    "artist_country",
    "lang",
    "release_year",
    "spotify_popularity",
    "danceability",
    "energy",
    "valence",
    "tempo",
    "duration_ms",
]
tracks = pd.read_parquet(DATA_DIR / "tracks_meta.parquet", columns=META_COLUMNS)
tracks = tracks.sort_values("m4a_id").reset_index(drop=True)
assert tracks.m4a_id.is_unique
track_ids = tracks.m4a_id.to_numpy(dtype=str)
id_to_idx = {track_id: i for i, track_id in enumerate(track_ids)}
train_qrels = pd.read_parquet(DATA_DIR / "train_qrels.parquet")
index_rows = index_inputs.merge(
    train_qrels[["query_id", "target_m4a_id"]], on="query_id", validate="one_to_one"
)
query_docs = (
    index_rows.drop_duplicates(["target_m4a_id", "normalized"])
    .groupby("target_m4a_id")["query"]
    .agg(" ".join)
)
tracks["query_document"] = tracks.m4a_id.map(query_docs).fillna("")
TEXT_COLUMNS = [
    "m4a_artist",
    "m4a_song",
    "m4a_album",
    "m4a_genres_full",
    "m4a_tags_full",
    "pseudo_caption",
    "lyrics",
    "artist_country",
    "lang",
    "release_year",
    "query_document",
]
documents = tracks[TEXT_COLUMNS].fillna("").astype(str).agg(" ".join, axis=1).tolist()
print(
    f"Треков: {len(tracks):,}; с train-запросами: {(tracks.query_document != '').sum():,}"
)
del index_inputs, train_inputs, query_docs
gc.collect()
artists = tracks.m4a_artist.fillna("").map(normalize).to_numpy()
popularity = (
    pd.to_numeric(tracks.spotify_popularity, errors="coerce").fillna(-1).to_numpy()
)
embedding_table = (
    pd.read_parquet(DATA_DIR / "track_embeddings.parquet")
    .set_index("m4a_id")
    .reindex(track_ids)
)
embeddings = np.stack(embedding_table.muq_embedding).astype(np.float32)
embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-8)
del embedding_table
metadata_documents = (
    tracks[[c for c in TEXT_COLUMNS if c not in ["lyrics", "query_document"]]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .tolist()
)
query_documents = tracks.query_document.tolist()


Треков: 64,016; с train-запросами: 28,335


## Готовые поисковые индексы

In [4]:
retriever = bm25s.BM25.load(str(BASE_DIR / "bm25_index"), mmap=True)
short_bm25 = bm25s.BM25.load(str(SEARCH_DIR / "metadata_bm25_index"), mmap=True)
assert np.array_equal(np.load(BASE_DIR / "track_ids.npy"), track_ids)
text_sources = {}
for name, folder, prefix in [
    ("tfidf", BASE_DIR, "tfidf"),
    ("query_tfidf", SEARCH_DIR, "query_tfidf"),
    ("metadata_char", SEARCH_DIR, "metadata_char"),
]:
    vectorizer = joblib.load(folder / f"{prefix}_vectorizer.joblib")
    matrix = load_npz(folder / f"{prefix}_tracks.npz").tocsr()
    text_sources[name] = (vectorizer, matrix.T.tocsr())


## История до границы теста

In [5]:
def top_indices(scores, k=MAX_K, positive=True):
    ids = np.flatnonzero(scores > 0) if positive else np.arange(len(scores))
    if len(ids) > k:
        threshold = np.partition(scores[ids], -k)[-k]
        larger = ids[scores[ids] > threshold]
        tied = ids[scores[ids] == threshold]
        ids = np.concatenate([larger, tied[: k - len(larger)]])
    return ids[np.lexsort((ids, -scores[ids]))].astype(np.int32)


def prepare_users(filename, user_ids, cutoff):
    table = pd.read_parquet(
        DATA_DIR / filename, columns=["user_id", "history", "history_ts"]
    )
    table = table[table.user_id.isin(user_ids)]
    cutoff = int(pd.Timestamp(cutoff, tz="UTC").timestamp())
    result = {}
    for row in tqdm(table.itertuples(index=False), total=len(table), desc=filename):
        counts = Counter()
        last = {}
        for track, timestamp in zip(row.history, row.history_ts):
            assert timestamp < cutoff
            idx = id_to_idx.get(track)
            if idx is not None:
                counts[idx] += 1
                last[idx] = int(timestamp)
        ids = np.array(list(counts), dtype=np.int32)
        freq = np.array([counts[i] for i in ids], dtype=np.float32)
        days = np.array([(cutoff - last[i]) / 86400 for i in ids])
        sources = {}
        for name, weights in [
            ("history", np.log1p(freq) * np.exp2(-days / 180)),
            ("history_recent", np.log1p(freq) * np.exp2(-days / 30)),
            ("history_frequency", freq),
        ]:
            sources[name] = ids[np.lexsort((ids, -weights))][:MAX_K]
        for name, weights in [
            ("audio_taste", freq),
            ("audio_recent", np.log1p(freq) * np.exp2(-days / 30)),
        ]:
            if not len(ids):
                sources[name] = np.array([], dtype=np.int32)
                continue
            centroid = np.einsum(
                "i,ij->j", weights.astype(np.float32), embeddings[ids], optimize=False
            )
            centroid /= max(float(np.linalg.norm(centroid)), 1e-8)
            scores = np.einsum("ij,j->i", embeddings, centroid, optimize=False)
            sources[name] = top_indices(scores)
        artist_weights = Counter()
        for idx, count in counts.items():
            if artists[idx]:
                artist_weights[artists[idx]] += count
        # Сначала более любимый артист, внутри артиста — Spotify popularity.
        scores = np.array([artist_weights.get(a, 0) for a in artists], dtype=np.float32)
        eligible = np.flatnonzero(scores > 0)
        sources["favorite_artists"] = eligible[
            np.lexsort((eligible, -popularity[eligible], -scores[eligible]))
        ][:MAX_K].astype(np.int32)
        sources["_counts"] = counts
        sources["_last"] = last
        sources["_cutoff"] = cutoff
        sources["_artist_counts"] = artist_weights
        result[row.user_id] = sources
    assert set(user_ids) <= set(result)
    return result


test_users = prepare_users(
    "test_public.parquet", set(test_inputs.user_id), "2020-02-20"
)
for user in test_users.values():
    counts = user["_counts"]
    indices = np.array(list(counts), dtype=np.int32)
    weights = np.array(list(counts.values()), dtype=np.float32)
    centroid = np.einsum("i,ij->j", weights, embeddings[indices], optimize=False)
    centroid /= max(float(np.linalg.norm(centroid)), 1e-8)
    user["_centroid"] = centroid
    user["_retrieval_scores"] = {
        i: float(
            np.log1p(c) * np.exp2(-(user["_cutoff"] - user["_last"][i]) / 86400 / 180)
        )
        for i, c in counts.items()
    }


## Объединение кандидатов и отбор 150

In [4]:
def retrieval_batches(inputs, users, cached=None):
    for start in range(0, len(inputs), BATCH_SIZE):
        batch = inputs.iloc[start : start + BATCH_SIZE]
        queries = batch["query"].tolist()
        vectors = {
            name: vectorizer.transform(queries)
            for name, (vectorizer, _) in text_sources.items()
        }
        scores = {
            name: (vectors[name] @ transpose).toarray()
            for name, (_, transpose) in text_sources.items()
        }
        source_batches = {}
        if cached is None:
            tokens = bm25s.tokenize(queries, stopwords="en", show_progress=False)
            for name, index in [("bm25", retriever), ("metadata_bm25", short_bm25)]:
                ids, values = index.retrieve(
                    tokens, k=TOP_K, sorted=True, n_threads=THREADS, show_progress=False
                )
                source_batches[name] = [
                    x[y > 0].astype(np.int32) for x, y in zip(ids, values)
                ]
        for offset, row in enumerate(batch.itertuples(index=False)):
            if cached is not None:
                sources = {name: cached[name][start + offset] for name in SOURCE_NAMES}
            else:
                sources = {
                    name: source_batches[name][offset] for name in source_batches
                }
                sources.update(
                    {
                        name: top_indices(values[offset])
                        for name, values in scores.items()
                    }
                )
                sources.update(
                    {
                        name: users[row.user_id][name]
                        for name in SOURCE_NAMES
                        if name not in sources
                    }
                )
            yield row, sources, {
                name: values[offset] for name, values in scores.items()
            }


def union_and_rrf(sources):
    union = np.unique(np.concatenate([sources[n] for n in SOURCE_NAMES])).astype(
        np.int32
    )
    rrf = np.zeros(len(track_ids), dtype=np.float32)
    for ids in sources.values():
        rrf[ids] += 1 / (RRF_CONSTANT + np.arange(1, len(ids) + 1))
    return union, rrf


def selection_scores(query_type, sources):
    policy = CAP_CONFIG["by_query_type"].get(str(query_type), CAP_CONFIG["fallback"])
    scores = np.zeros(len(track_ids), dtype=np.float32)
    text_names = {"bm25", "tfidf", "metadata_bm25", "metadata_char", "query_tfidf"}
    for name in policy["generators"]:
        ids = sources[name]
        weight = policy["text_weight"] if name in text_names else 1
        scores[ids] += weight / (policy["rrf_constant"] + np.arange(1, len(ids) + 1))
    return scores


def cap_candidates(query_type, sources, union):
    scores = selection_scores(query_type, sources)
    eligible = union[scores[union] > 0]
    if len(eligible) > CANDIDATE_LIMIT:
        threshold = np.partition(scores[eligible], -CANDIDATE_LIMIT)[-CANDIDATE_LIMIT]
        above = eligible[scores[eligible] > threshold]
        tied = eligible[scores[eligible] == threshold]
        eligible = np.concatenate([above, tied[: CANDIDATE_LIMIT - len(above)]])
    ids = np.sort(eligible).astype(np.int32)
    assert len(ids) <= CANDIDATE_LIMIT and len(ids) == len(np.unique(ids))
    return ids


## Те же 58 фичей, что при обучении

In [7]:
def word_set(text):
    return set(normalize(text).split())


def bigram_set(text):
    words = normalize(text).split()
    return set(zip(words, words[1:]))


def trigram_set(text):
    text = normalize(text)
    return {text[i : i + 3] for i in range(max(len(text) - 2, 0))}


def dice(left, right):
    return 2 * len(left & right) / max(len(left) + len(right), 1)


word_vectorizer = CountVectorizer(
    analyzer=lambda x: list(word_set(x)), binary=True, dtype=np.float32
)
word_vectorizer.fit(documents)
coverage_documents = {
    "document_coverage": documents,
    "title_coverage": tracks.m4a_song.fillna("").tolist(),
    "artist_coverage": tracks.m4a_artist.fillna("").tolist(),
    "tag_coverage": tracks[["m4a_genres_full", "m4a_tags_full"]]
    .fillna("")
    .agg(" ".join, axis=1)
    .tolist(),
    "lyrics_coverage": tracks.lyrics.fillna("").tolist(),
}
coverage_matrices = {
    name: word_vectorizer.transform(texts).tocsr()
    for name, texts in coverage_documents.items()
}
word_lengths = {
    name: np.asarray(matrix.sum(axis=1)).ravel()
    for name, matrix in coverage_matrices.items()
}
normalized_titles = tracks.m4a_song.fillna("").map(normalize).to_numpy()
title_trigrams = [trigram_set(x) for x in normalized_titles]
artist_trigrams = [trigram_set(x) for x in artists]
lyrics_bigrams = [bigram_set(x) for x in tracks.lyrics.fillna("")]
del documents, metadata_documents, query_documents, index_rows, coverage_documents
popularity = popularity.astype(np.float32)
gc.collect()


def feature_frame(row, sources, text_scores, user, ids, union, all_rrf):
    query_text = normalize(row.query)
    query_words = word_set(row.query)
    query_bigram = bigram_set(row.query)
    query_trigram = trigram_set(row.query)
    tokens = bm25s.tokenize(
        [row.query], stopwords="en", return_ids=False, show_progress=False
    )[0]
    bm25 = (
        retriever.get_scores(tokens)
        if tokens
        else np.zeros(len(track_ids), dtype=np.float32)
    )
    counts = user["_counts"]
    total_count = max(sum(counts.values()), 1)
    last = user["_last"]
    history = user["_retrieval_scores"]
    ranks = {
        name: {int(idx): rank for rank, idx in enumerate(values, 1)}
        for name, values in sources.items()
    }
    b = bm25[ids]
    t = text_scores["tfidf"][ids]
    h = np.array([history.get(int(i), 0) for i in ids], dtype=np.float32)
    old_rrf = np.zeros(len(ids), dtype=np.float32)
    for name in ["bm25", "tfidf", "history"]:
        old_rrf += np.array(
            [
                1 / (RRF_CONSTANT + ranks[name][int(i)]) if int(i) in ranks[name] else 0
                for i in ids
            ],
            dtype=np.float32,
        )
    values = {
        "bm25_score": b,
        "bm25_rank": [ranks["bm25"].get(int(i), TOP_K + 1) for i in ids],
        "bm25_relative_score": b
        / max(float(bm25[sources["bm25"]].max()) if len(sources["bm25"]) else 0, 1e-8),
        "tfidf_score": t,
        "history_retrieval_score": h,
        "rrf_score": old_rrf,
        "spotify_popularity": popularity[ids],
        "history_track_log_count": [np.log1p(counts.get(int(i), 0)) for i in ids],
        "history_artist_share": [
            user["_artist_counts"].get(artists[i], 0) / total_count for i in ids
        ],
        "history_track_recency_days": [
            (user["_cutoff"] - last[int(i)]) / 86400 if int(i) in last else -1
            for i in ids
        ],
        "history_audio_cosine": np.einsum(
            "ij,j->i", embeddings[ids], user["_centroid"], optimize=False
        ),
        "tfidf_rank": [ranks["tfidf"].get(int(i), TOP_K + 1) for i in ids],
        "tfidf_relative_score": t / max(float(text_scores["tfidf"][union].max()), 1e-8),
        "history_rank": [ranks["history"].get(int(i), TOP_K + 1) for i in ids],
        "query_token_count": len(query_words),
        "title_phrase_match": [
            float(
                bool(normalized_titles[i])
                and f" {normalized_titles[i]} " in f" {query_text} "
            )
            for i in ids
        ],
        "artist_phrase_match": [
            float(bool(artists[i]) and f" {artists[i]} " in f" {query_text} ")
            for i in ids
        ],
        "lyrics_bigram_coverage": [
            len(query_bigram & lyrics_bigrams[i]) / max(len(query_bigram), 1)
            for i in ids
        ],
        "title_trigram_dice": [dice(query_trigram, title_trigrams[i]) for i in ids],
        "artist_trigram_dice": [dice(query_trigram, artist_trigrams[i]) for i in ids],
        "bm25_score_share": b / max(float(bm25[union].sum()), 1e-8),
        "tfidf_score_share": t / max(float(text_scores["tfidf"][union].sum()), 1e-8),
        "history_score_normalized": h / max(max(history.values(), default=0), 1e-8),
        "all_rrf_score": all_rrf[ids],
    }
    query_vector = word_vectorizer.transform([row.query])
    for name, matrix in coverage_matrices.items():
        overlap = (matrix[ids] @ query_vector.T).toarray().ravel()
        values[name] = overlap / max(len(query_words), 1)
        if name == "title_coverage":
            values["title_token_recall"] = overlap / np.maximum(
                word_lengths[name][ids], 1
            )
        if name == "artist_coverage":
            values["artist_token_recall"] = overlap / np.maximum(
                word_lengths[name][ids], 1
            )
        if name == "tag_coverage":
            values["tag_jaccard"] = overlap / np.maximum(
                len(query_words) + word_lengths[name][ids] - overlap, 1
            )
    source_count = np.zeros(len(ids), dtype=np.float32)
    for name in SOURCE_NAMES:
        rank = np.array(
            [ranks[name].get(int(i), TOP_K + 1) for i in ids], dtype=np.float32
        )
        hit = (rank <= TOP_K).astype(np.float32)
        values[f"{name}_source_rank"] = rank
        values[f"{name}_source_hit"] = hit
        source_count += hit
    values["all_source_count"] = source_count
    selection = selection_scores(row.query_type, sources)[ids]
    order = np.lexsort((ids, -selection))
    selection_ranks = np.empty(len(ids), dtype=np.float32)
    selection_ranks[order] = np.arange(1, len(ids) + 1)
    values["selection_rrf_score"] = selection
    values["selection_rank"] = selection_ranks
    frame = pd.DataFrame(values, columns=NUMERIC_FEATURES, dtype=np.float32)
    frame["query_type"] = str(row.query_type)
    assert np.isfinite(frame[NUMERIC_FEATURES].to_numpy()).all()
    return frame[FEATURES]


## Исключение референса из запроса

In [8]:
# Сопоставляем референс с каталогом только по тексту запроса.
# Важно: название должно начинаться сразу после like / similar to,
# иначе общее слово «more» могло бы совпасть с посторонним названием.
title_lookup = {}
for idx, title in enumerate(normalized_titles):
    if title:
        title_lookup.setdefault(title, []).append(idx)
artist_lookup = set(artists) - {""}


def longest_prefix(text, lookup):
    words = text.split()
    for end in range(len(words), 0, -1):
        phrase = " ".join(words[:end])
        if phrase in lookup:
            return phrase
    return ""


def reference_artists(row):
    if row.query_type != "similar_to":
        return set()
    text = normalize(row.query)
    excluded = set()
    for match in re.finditer(r"\b(?:by|not|excluding|except)\s+(.+)", text):
        rest = match.group(1)
        if rest.split()[0] in {"him", "her", "them", "that", "this", "the", "from"}:
            continue
        artist = longest_prefix(rest, artist_lookup)
        if artist:
            excluded.add(artist)
    marker = re.search(r"\b(?:similar to|like)\s+(.+)", text)
    if marker:
        title = longest_prefix(marker.group(1), title_lookup)
        if title:
            candidates = {artists[i] for i in title_lookup[title]} - {""}
            # Явно названный артист снимает неоднозначность названия.
            explicitly_named = {a for a in candidates if f" {a} " in f" {text} "}
            if explicitly_named:
                excluded.update(explicitly_named)
            elif len(candidates) == 1:
                excluded.update(candidates)
    return excluded


## Короткий ответ про первый трек

In [5]:
# Все утверждения о музыке берём из метаданных, о вкусе — из history.
def clean_text(value, limit=12):
    if pd.isna(value):
        return ""
    return " ".join(str(value).split()[:limit])


def russian_request(query):
    # Кириллическая цитата внутри английского запроса не меняет язык ответа.
    return bool(re.search("[а-яА-Я]", query)) and not re.search(
        r"\b(find|song|music|track|play|with|by|that|like|lyrics|rap)\b",
        query,
        flags=re.I,
    )


def make_response(row, idx, user):
    track = tracks.iloc[idx]
    title = clean_text(track.m4a_song)
    artist = clean_text(track.m4a_artist)
    genres = list(
        dict.fromkeys(
            x.strip()
            for x in str(track.m4a_genres_full).split(",")
            if x.strip() and x.strip().lower() not in {"nan", "none"}
        )
    )[:3]
    tags = list(
        dict.fromkeys(
            genres
            + [
                x.strip()
                for x in str(track.m4a_tags_full).split(",")
                if x.strip() and x.strip().lower() not in {"nan", "none"}
            ]
        )
    )
    query_text = f" {normalize(row.query)} "
    matching = [x for x in tags if f" {normalize(x)} " in query_text][:3]
    familiar = bool(user["_artist_counts"].get(artists[idx], 0))
    top_artist = user["_artist_counts"].most_common(1)
    history_artist = ""
    if top_artist:
        historical_idx = next(
            (i for i in user["_counts"] if artists[i] == top_artist[0][0]), None
        )
        if historical_idx is not None:
            history_artist = clean_text(tracks.iloc[historical_idx].m4a_artist)
    if russian_request(row.query):
        text = f"Предлагаю «{title}» — {artist}."
        if genres:
            text += " В метаданных указаны жанры: " + ", ".join(genres) + "."
        if matching:
            text += " С запросом совпадают теги: " + ", ".join(matching) + "."
        elif normalized_titles[idx] and f" {normalized_titles[idx]} " in query_text:
            text += " Название трека встречается в вашем запросе."
        else:
            text += " Выбран с учётом текста запроса и вашей истории прослушиваний."
        if familiar:
            text += (
                " Вы уже слушали этого исполнителя; это учитывает ваш музыкальный вкус."
            )
        elif history_artist:
            text += f" Среди исполнителей в вашей истории — {history_artist}; эта история также учтена при выборе."
    else:
        text = f"I recommend “{title}” by {artist}."
        if genres:
            text += " Its metadata lists " + ", ".join(genres) + "."
        if matching:
            text += (
                " It shares these tags with your request: " + ", ".join(matching) + "."
            )
        elif normalized_titles[idx] and f" {normalized_titles[idx]} " in query_text:
            text += " The track title appears in your request."
        else:
            text += " I selected it using your request and listening history."
        if familiar:
            text += " You have listened to this artist before, so the choice reflects that part of your taste."
        elif history_artist:
            text += f" Your history includes {history_artist}; that listening context also informed this choice."
    assert 0 < len(text.split()) <= 120
    return text


## Прогнозы для полного public test

In [7]:
import time

schema = pa.schema(
    [
        ("query_id", pa.string()),
        ("top20", pa.list_(pa.string())),
        ("response", pa.string()),
    ]
)
# Части — внутренние контрольные точки. Итогом будет один parquet.
existing = sorted(PARTS_DIR.glob("part_*.parquet"))
completed = sum(pq.ParquetFile(path).metadata.num_rows for path in existing)
assert completed <= len(test_inputs)
if completed:
    saved_ids = [
        qid
        for path in existing
        for qid in pq.read_table(path, columns=["query_id"])["query_id"].to_pylist()
    ]
    assert saved_ids == test_inputs.query_id.iloc[:completed].tolist()
    del saved_ids
remaining = test_inputs.iloc[completed:].reset_index(drop=True)
stream = retrieval_batches(remaining, test_users)
records = []
part_number = len(existing)
processed = completed
expanded_count = 0
start_time = time.time()
while True:
    batch = list(islice(stream, BATCH_SIZE))
    if not batch:
        break
    frames, info = [], []
    for row, sources, text_scores in batch:
        user = test_users[row.user_id]
        union, rrf = union_and_rrf(sources)
        ids = cap_candidates(row.query_type, sources, union)
        excluded = reference_artists(row)
        valid = np.array([artists[i] not in excluded for i in ids])
        # Сначала сохраняем обученную политику. Если после фильтра менее 20,
        # добавляем кандидатов из исходного объединения генераторов.
        if valid.sum() < 20:
            extra = [i for i in union if artists[i] not in excluded]
            ids = np.unique(np.concatenate([ids, np.asarray(extra, dtype=np.int32)]))
            valid = np.array([artists[i] not in excluded for i in ids])
            expanded_count += 1
        assert valid.sum() >= 20, row.query_id
        frames.append(feature_frame(row, sources, text_scores, user, ids, union, rrf))
        info.append((row, ids, valid, user))
    scores = model.predict(pd.concat(frames, ignore_index=True), thread_count=THREADS)
    offset = 0
    for row, ids, valid, user in info:
        values = scores[offset : offset + len(ids)]
        offset += len(ids)
        eligible = np.flatnonzero(valid)
        order = eligible[np.lexsort((ids[eligible], -values[eligible]))][:20]
        top_ids = ids[order]
        records.append(
            {
                "query_id": row.query_id,
                "top20": track_ids[top_ids].tolist(),
                "response": make_response(row, int(top_ids[0]), user),
            }
        )
    processed += len(batch)
    if len(records) >= 1024 or processed == len(test_inputs):
        part = PARTS_DIR / f"part_{part_number:05d}.parquet"
        temporary = part.with_suffix(".tmp")
        pq.write_table(
            pa.Table.from_pylist(records, schema=schema), temporary, compression="zstd"
        )
        temporary.replace(part)
        part_number += 1
        records.clear()
        elapsed = time.time() - start_time
        progress = {
            "completed": processed,
            "total": len(test_inputs),
            "elapsed_seconds": round(elapsed, 1),
            "queries_per_second": round((processed - completed) / max(elapsed, 1), 2),
            "expanded_candidate_queries_this_run": expanded_count,
        }
        (SUBMIT_DIR / "progress.json").write_text(json.dumps(progress, indent=2))
        print(progress, flush=True)
    del frames, info, batch
assert processed == len(test_inputs)
temporary = SUBMISSION_PATH.with_suffix(".tmp.parquet")
# Более крупные row groups уменьшают размер файла для загрузки.
with pq.ParquetWriter(temporary, schema=schema, compression="zstd") as writer:
    tables = []
    for part in sorted(PARTS_DIR.glob("part_*.parquet")):
        tables.append(pq.read_table(part, schema=schema))
        if len(tables) == 32:
            writer.write_table(pa.concat_tables(tables), row_group_size=32768)
            tables.clear()
    if tables:
        writer.write_table(pa.concat_tables(tables), row_group_size=32768)
temporary.replace(SUBMISSION_PATH)
print("Saved:", SUBMISSION_PATH)


Saved: /Users/aleksei/Desktop/projects_vs_code/baseline_sota2026/submission_public.parquet


## Проверка формата и примеры

In [8]:
submission = pd.read_parquet(SUBMISSION_PATH)
assert submission.columns.tolist() == ["query_id", "top20", "response"]
assert submission.query_id.is_unique
assert submission.query_id.tolist() == test_inputs.query_id.tolist()
known_ids = set(track_ids)
assert submission.top20.map(
    lambda x: len(x) == 20 and len(set(x)) == 20 and set(x) <= known_ids
).all()
word_counts = submission.response.map(lambda x: len(x.split()))
assert word_counts.between(1, 120).all()
report = {
    "split": "test_public",
    "rows": len(submission),
    "tracks_per_query": 20,
    "maximum_response_words": int(word_counts.max()),
    "model": str(OUT_DIR / "catboost_ranker.cbm"),
    "response_generation": "metadata/history templates",
    "public_qrels_used": False,
    "submission": str(SUBMISSION_PATH),
}
(SUBMIT_DIR / "validation.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False)
)
print(report)
display(submission.head(5))


{'split': 'test_public', 'rows': 273137, 'tracks_per_query': 20, 'maximum_response_words': 54, 'model': '/Users/aleksei/Desktop/projects_vs_code/baseline_sota2026/adaptive150_catboost_artifacts/catboost_ranker.cbm', 'response_generation': 'metadata/history templates', 'public_qrels_used': False, 'submission': '/Users/aleksei/Desktop/projects_vs_code/baseline_sota2026/submission_public.parquet'}


,query_id,top20,response
0,3fb0b12f6cef99e5,"[CSeYHcWuYRSvyDY7, 7RpMnPDp9XTk0ZHI, kZkC2QqCK...",I recommend “Tempest” by Deftones. Its metadat...
1,add92d604a1792c4,"[UU4ihf3afIkSItxV, jjbWpQb4ixMJUeeY, dkjlBYXTh...",I recommend “Dystopia” by TesseracT. Its metad...
2,40ec499b8150d446,"[xjPbsW9OgACGpbD5, 5FGBYu8CM1MlXokl, CAybWF57A...",I recommend “Hexes” by TesseracT. Its metadata...
3,967545d36bf70231,"[m0jTS5YbHL2L5abU, kOsa3hHQPnDcCXiR, 1CCSNgcHs...",I recommend “Utopia” by TesseracT. Its metadat...
4,f9c3a89380ea63fa,"[AeKkHRHqBcGAKExX, dMaVytJfMguUzrgj, e9v6XWPgG...",I recommend “Yes It Is” by The Beatles. Its me...
